In [9]:
import json
import data_utils
import conceptset_utils

In [10]:
"""
CLASS_SIM_CUTOFF: Concenpts with cos similarity higher than this to any class will be removed
OTHER_SIM_CUTOFF: Concenpts with cos similarity higher than this to another concept will be removed
MAX_LEN: max number of characters in a concept

PRINT_PROB: what percentage of filtered concepts will be printed
"""

CLASS_SIM_CUTOFF = 0.85
OTHER_SIM_CUTOFF = 0.9
MAX_LEN = 30
PRINT_PROB = 1

dataset = "chestxray"
device = "cuda"

save_name = "data/concept_sets/{}_filtered.txt".format(dataset)

In [11]:
#EDIT these to use the initial concept sets you want

with open("data/concept_sets/gpt3_init/gpt35-turbo_{}_important.json".format(dataset), "r") as f:
    important_dict = json.load(f)
with open("data/concept_sets/gpt3_init/gpt35-turbo_{}_superclass.json".format(dataset), "r") as f:
    superclass_dict = json.load(f)
with open("data/concept_sets/gpt3_init/gpt35-turbo_{}_around.json".format(dataset), "r") as f:
    around_dict = json.load(f)
    
with open(data_utils.LABEL_FILES[dataset], "r") as f:
    classes = f.read().split("\n")

In [12]:
concepts = set()

for values in important_dict.values():
    concepts.update(set(values))

for values in superclass_dict.values():
    concepts.update(set(values))
    
for values in around_dict.values():
    concepts.update(set(values))

print(len(concepts))

160


In [13]:
concepts = conceptset_utils.remove_too_long(concepts, MAX_LEN, PRINT_PROB)

32 increased opacity on chest x-ray
44 pain or discomfort in the area of the hernia
42 loss of lung markings in the affected area
49 surrounding ground glass opacity or consolidation
47 enlarged cardiac silhouette on chest radiograph
33 pleural plaques or calcifications
57 homogeneous increase in pulmonary parenchymal attenuation
79 -protrusion of an organ or tissue through a weakened area in the abdominal wall
32 increased opacity in lung fields
46 -lack of lung markings at the edge of the lung
35 blunting of the costophrenic angles
36 ill-defined margins of lung markings
41 shift of mediastinum to the affected side
32 -honeycombing pattern on imaging
40 may show enhancement on contrast imaging
73 -protrusion of an organ or tissue through an abnormal opening in the body
32 solid or ground-glass appearance
48 homogeneous opacity in the affected lung segment
39 -thickening and scarring of lung tissue
43 hazy opacity on chest radiograph or CT scan
44 increased opacity in the affected hem

In [14]:
concepts = conceptset_utils.filter_too_similar_to_cls(concepts, classes, CLASS_SIM_CUTOFF, device, PRINT_PROB)

114
114
Class:Infiltration - Concept:infiltrates, sim:0.884 - Deleting infiltrates

Class:Fibrosis - Concept:pulmonary fibrosis, sim:0.858 - Deleting pulmonary fibrosis

Class:Pneumonia - Concept:-pneumonia, sim:0.901 - Deleting -pneumonia

Class:Pleural_thickening - Concept:-pleural effusion, sim:0.859 - Deleting -pleural effusion

Class:Pleural_thickening - Concept:-thickening of the pleura, sim:0.881 - Deleting -thickening of the pleura

Class:Pleural_thickening - Concept:pleural effusion, sim:0.876 - Deleting pleural effusion

Class:Pleural_thickening - Concept:pleural line visible, sim:0.851 - Deleting pleural line visible

Class:Pleural_thickening - Concept:pleural space, sim:0.873 - Deleting pleural space

Class:Hernia - Concept:hernia sac, sim:0.920 - Deleting hernia sac

105


In [15]:
concepts = conceptset_utils.filter_too_similar(concepts, OTHER_SIM_CUTOFF, device, PRINT_PROB)

- fever - fever , sim:0.9118 - Deleting - fever
-CT scan - CT scan , sim:0.9522 - Deleting CT scan
-cancer - -tumor , sim:0.9014 - Deleting -cancer
-collapsed lung - -lung collapse , sim:0.9377 - Deleting -collapsed lung
-fluid buildup - fluid buildup , sim:0.9588 - Deleting fluid buildup
-heart condition - -medical condition , sim:0.9017 - Deleting -medical condition
-increased lung density - decreased lung density , sim:0.9250 - Deleting -increased lung density
-lung collapse - -partial lung collapse , sim:0.9425 - Deleting -lung collapse
-lung disease - -pulmonary condition , sim:0.9145 - Deleting -lung disease
-lungs - lungs , sim:0.9136 - Deleting lungs
-pulmonary condition - -pulmonary disorder , sim:0.9502 - Deleting -pulmonary disorder
-pulmonary condition - -respiratory condition , sim:0.9165 - Deleting -pulmonary condition
-respiratory condition - respiratory condition , sim:0.9655 - Deleting respiratory condition
-swelling - swelling , sim:0.9380 - Deleting swelling
-tumor -

In [16]:
with open(save_name, "w") as f:
    f.write(concepts[0])
    for concept in concepts[1:]:
        f.write("\n" + concept)